# WEMA — Test Suite Walkthrough

**Author:** Victoria Fakunle
**Institution:** African Leadership University
**Purpose:** a guided, file-by-file walk through WEMA's real `pytest` suite — what each test file proves about the shipped code, run live against the actual `src/` modules (not reimplemented copies).

This notebook answers a specific defense question: *"you claim automated tests — show me."* Every cell below executes the real test files in [`tests/`](../tests/) against the real `src/` modules with `subprocess`, and the output shown is what that run actually printed — not typed by hand.

**Why these tests don't call Groq / Twilio / Deepgram / Azure:** by design. They test the deterministic logic around the AI call — keyword routing, regex safety nets, state extraction, Haversine ranking, session isolation — not the LLM's own output, which is non-deterministic and evaluated separately in [`WEMA_Testing_and_Evaluation.ipynb`](WEMA_Testing_and_Evaluation.ipynb) (68 clinician-reviewed scenarios, LLM-judge scored). Unit tests prove the guardrails around the model behave correctly every time; the evaluation notebook proves the model's actual answers are clinically sound most of the time. Both are needed — neither substitutes for the other.


---
## 0. Setup

`tests/conftest.py` inserts `src/` onto `sys.path` so every test file can `from sms import ...` / `from rag import ...` / `from prompt import ...` directly against the real production modules — a failing assertion here means the *shipped* code broke, not a stale test double.


In [1]:
!python --version
!python -m pytest --version

3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]
pytest 8.3.2


---
## 1. `tests/test_prompt.py` — fallback & conversational-intent routing (17 tests)

Tests the canned-response layer in `src/prompt.py` — what a caller hears when the system deliberately **skips** the LLM (small talk) or **falls back** after a failure. Notably:

- `get_emergency_fallback()` keyword-routes 6 scenario types (bleeding, seizure, cord, not-breathing, ectopic, default) and is asserted to **never say "massage" for bleeding without a birth mention** (dangerous for placenta praevia) and **never name a single medication** across any scenario — a hard safety property, checked against a banned-word list (`paracetamol`, `ibuprofen`, `ginger tea`, `vitamin b6`, `misoprostol`).
- `is_conversational()` is asserted to catch "hi"/"thank you"/"ok" but *not* misfire on a real symptom sentence.


In [2]:
!python -m pytest tests/test_prompt.py -v --no-header

============================= test session starts =============================
collecting ... collected 17 items

tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I am bleeding heavily after birth-massage] PASSED [  5%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[She is having seizures she is pregnant-left side] PASSED [ 11%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[Baby is not breathing after delivery-dry] PASSED [ 17%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I need help with my pregnancy-left side] PASSED [ 23%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I am 6 months pregnant and I am bleeding-do not press] PASSED [ 29%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I no fit stand up, my belly dey pain me-left side] PASSED [ 35%]
tests/test_prompt.py::test_get_emergency_fallback_pregnancy_bleeding_never_says_massage PASSED [ 41%]
tests/test_prompt.py:

---
## 2. `tests/test_rag_safety_net.py` — the parts of `src/rag.py` testable without a live Groq call (14 tests)

Importing `rag.py` pulls in the full ML stack (`langchain`/`torch`/`sentence-transformers`) but makes **no network call** at import time or in these functions, so this suite runs fully offline:

- `_secondary_pph_risk()` — the regex that injects an extra safety note when bleeding is described as starting/continuing *days or weeks after* birth (secondary PPH, where belly massage is dangerous). One test (`test_secondary_pph_risk_misses_spelled_out_numbers`) deliberately documents a **known, currently-open gap**: the regex matches digit forms ("2 weeks ago") but not spelled-out numbers ("two weeks ago") — asserted as `False` on purpose, so this can never silently regress further without the test itself changing.
- `_is_pidgin()` — the keyword detector that routes Nigerian Pidgin callers away from the LLM entirely.
- `test_ask_wema_pidgin_bypasses_generation_entirely` calls the real `ask_wema()` with `vectorstore=None` — if generation were reached at all, it would crash on the `None`. It doesn't crash, which is the proof the Pidgin path never touches retrieval or the LLM.
- `test_ask_wema_pidgin_no_fit_is_not_a_seizure` — Pidgin "no fit" (= *cannot*) must not trigger the seizure/fits response just because it contains "fit".


In [3]:
!python -m pytest tests/test_rag_safety_net.py -v --no-header

============================= test session starts =============================
collecting ... collected 14 items

tests/test_rag_safety_net.py::test_secondary_pph_risk_detects_days_after_birth_bleeding PASSED [  7%]
tests/test_rag_safety_net.py::test_secondary_pph_risk_detects_weeks_since_delivery PASSED [ 14%]
tests/test_rag_safety_net.py::test_secondary_pph_risk_misses_spelled_out_numbers PASSED [ 21%]
tests/test_rag_safety_net.py::test_secondary_pph_risk_false_for_immediate_postpartum_bleeding PASSED [ 28%]
tests/test_rag_safety_net.py::test_secondary_pph_risk_false_when_not_pregnancy_related PASSED [ 35%]
tests/test_rag_safety_net.py::test_secondary_pph_risk_false_for_bleeding_without_birth_mention PASSED [ 42%]
tests/test_rag_safety_net.py::test_classify_risk_high_for_alerting_language PASSED [ 50%]
tests/test_rag_safety_net.py::test_classify_risk_medium_for_routine_visit_language PASSED [ 57%]
tests/test_rag_safety_net.py::test_classify_risk_low_for_reassurance_only PASSED [ 64%

---
## 3. `tests/test_sms.py` — state extraction, Haversine ranking, provider lookup (21 tests)

Tests `src/sms.py` end to end minus the actual Twilio network send:

- `extract_state()` — keyword-matches spoken text against 21 Nigerian states (e.g. "Port Harcourt" → `Rivers`, "Maiduguri" → `Borno`), and is asserted to return `None` (not a wrong guess) when no place name is mentioned at all.
- `haversine_distance()` / ranking — asserted against 3 real Lagos-area hospital coordinates, checking both correct sort order and a sanity distance bound.
- `ProviderDirectory` — the class wrapping the provider CSV. Tests use a **throwaway fixture CSV** (`tmp_path`), not the real `data/providers.csv`, so these are independent of whatever's currently in the live demo data — including one row deliberately malformed (`not-a-number` for latitude) to prove `load()` skips unparseable rows instead of crashing.


In [4]:
!python -m pytest tests/test_sms.py -v --no-header

============================= test session starts =============================
collecting ... collected 21 items

tests/test_sms.py::test_should_trigger_sms[Help is being alerted. Get to a health facility now.-True] PASSED [  4%]
tests/test_sms.py::test_should_trigger_sms[Alerting the nearest doctor to you right now.-True] PASSED [  9%]
tests/test_sms.py::test_should_trigger_sms[I am alerting a doctor near you now.-True] PASSED [ 14%]
tests/test_sms.py::test_should_trigger_sms[Lie on your left side and rest.-False] PASSED [ 19%]
tests/test_sms.py::test_should_trigger_sms[Drink plenty of water and monitor your symptoms.-False] PASSED [ 23%]
tests/test_sms.py::test_should_trigger_sms[-False] PASSED                [ 28%]
tests/test_sms.py::test_extract_state[I am in Lagos, I am bleeding heavily-Lagos] PASSED [ 33%]
tests/test_sms.py::test_extract_state[I am calling from Kano-Kano] PASSED [ 38%]
tests/test_sms.py::test_extract_state[I live in Port Harcourt-Rivers] PASSED [ 42%]
tests/test

---
## 4. `tests/test_session_store.py` — per-call state isolation (6 tests)

Tests `src/session_store.py` in complete isolation from Flask/Twilio/ChromaDB. The one that matters most for correctness under concurrent calls: `test_sessions_are_isolated_per_call_sid` — mutating one call's session must never leak into another call's session dict. `test_stores_are_independent_instances` guards against a subtle Python bug class (accidentally shared mutable class-level defaults instead of per-instance state).


In [5]:
!python -m pytest tests/test_session_store.py -v --no-header

============================= test session starts =============================
collecting ... collected 6 items

tests/test_session_store.py::test_get_session_creates_default_state_once PASSED [ 16%]
tests/test_session_store.py::test_sessions_are_isolated_per_call_sid PASSED [ 33%]
tests/test_session_store.py::test_response_ready_set_and_pop PASSED      [ 50%]
tests/test_session_store.py::test_pop_response_ready_missing_call_sid_returns_none PASSED [ 66%]
tests/test_session_store.py::test_audio_cache_roundtrip PASSED           [ 83%]
tests/test_session_store.py::test_stores_are_independent_instances PASSED [100%]

============================== 6 passed in 0.10s ==============================


---
## 5. `tests/test_legal_pages.py` — the public `/privacy` and `/terms` routes (2 tests)

The only file that imports the real Flask `app` object rather than a bare module — which means it must first stub out `load_vectorstore` and provide dummy Twilio credentials (see the docstring in the file) so importing `app.py` doesn't try to build a Twilio client with `None` credentials or download the embedding model during a test run. `/terms` is asserted to redirect to `/privacy`, not duplicate it.


In [6]:
!python -m pytest tests/test_legal_pages.py -v --no-header

============================= test session starts =============================
collecting ... collected 2 items

tests/test_legal_pages.py::test_privacy_returns_200_with_known_string PASSED [ 50%]
tests/test_legal_pages.py::test_terms_redirects_to_privacy PASSED        [100%]

============================== warnings summary ===============================
wema_prod_env\Lib\site-packages\deepgram\clients\listen\v1\websocket\async_client.py:12
  C:\Users\pampam\WEMA\WEMA-Women-s-Emergency-Medical-AI\wema_prod_env\Lib\site-packages\deepgram\clients\listen\v1\websocket\async_client.py:12: DeprecationWarning: websockets.client.WebSocketClientProtocol is deprecated
    from websockets.client import WebSocketClientProtocol

wema_prod_env\Lib\site-packages\websockets\legacy\__init__.py:6
  C:\Users\pampam\WEMA\WEMA-Women-s-Emergency-Medical-AI\wema_prod_env\Lib\site-packages\websockets\legacy\__init__.py:6: DeprecationWarning: websockets.legacy is deprecated; see https://websockets.readthedoc

---
## 6. Full suite, one run

All 5 files together — this is the number that matters for the defense: how many automated, `assert`-based, CI-able tests currently pass against the real shipped code, right now.


In [7]:
!python -m pytest tests/ -v --no-header

============================= test session starts =============================
collecting ... collected 60 items

tests/test_legal_pages.py::test_privacy_returns_200_with_known_string PASSED [  1%]
tests/test_legal_pages.py::test_terms_redirects_to_privacy PASSED        [  3%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I am bleeding heavily after birth-massage] PASSED [  5%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[She is having seizures she is pregnant-left side] PASSED [  6%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[Baby is not breathing after delivery-dry] PASSED [  8%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I need help with my pregnancy-left side] PASSED [ 10%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I am 6 months pregnant and I am bleeding-do not press] PASSED [ 11%]
tests/test_prompt.py::test_get_emergency_fallback_keyword_routing[I no fit stand up, my be

---
## 7. What this suite deliberately does *not* cover

Being honest about scope matters more than a big number:

- **No live LLM call is tested here.** Whether `ask_wema()`'s *generated* clinical guidance is correct is a judgment call, not a deterministic assertion — that's what the separate 68-scenario, LLM-judge-scored evaluation notebook is for.
- **No live Twilio/Deepgram/Azure network call is tested here.** These tests exercise the logic *around* those calls (state extraction, fallback text, session state), not the calls themselves — there's no automated test today that dials the real number end-to-end; `src/test_call.py` and `src/test_deepgram.py` are manual connectivity checks, not part of `pytest tests/`.
- **No concurrency/load test.** `test_session_store.py` proves two `SessionStore` *instances* don't leak state into each other, but nothing here simulates two simultaneous real calls hitting the single Gunicorn worker (`--workers 1` in the `Dockerfile`) at once.
